## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'LOCAL_PETCARE_MARKETPLACE'
LIVE_TEST_LEVEL = 'ONE_CLICK'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'LOCAL_PETCARE_MARKETPLACE',
 'testLevel': 'ONE_CLICK',
 'domain': 'Local service marketplace',
 'expectedStructuralFeatures': ['중개', '파트너 제공자', '예약과 결제', '개인정보']}

## 06. Idea Brief Derivation


In [7]:
engine._reset()
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': True, 'interpretationPresent': bool(seed.interpretation)})

{'ideaCallComplete': True, 'interpretationPresent': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json'))
assert idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 반려동물 보호자와 펫시터를 연결하는 서비스입니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context))

,항목,AI 이해 결과
0,interpretedProblem,보호자가 가까운 지역에서 믿을 수 있는 돌봄 제공자를 빠르게 찾고 비교하기 어렵다.
1,interpretedTargetUsers,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자
2,usageContext,반려동물 돌봄 서비스 제공
3,industryCategory,반려동물 서비스
4,researchScope,지역 기반 펫시터 서비스
5,conciseIdeaDefinition,반려동물 보호자와 검증된 지역 펫시터를 연결하는 플랫폼
6,targetRegionInterpretation,구체적인 지역 정보는 제공되지 않음
7,relevantKnownCompetitorContext,경쟁자 정보는 제공되지 않음


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context))

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 서비스는 반려동물 보호자와 검증된 지역 펫시터를 연결하여 예약, 결제 및 이용 후기를 제공하는 플랫폼입니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed)
display(show_seed_analysis(analysis))

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json'))

{'problemCore': '보호자가 가까운 지역에서 믿을 수 있는 돌봄 제공자를 빠르게 찾고 비교하기 어렵다.',
 'targetCore': '출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자',
 'useContexts': ['반려동물 돌봄 서비스 제공'],
 'intentComponents': ['반려동물 보호자와 검증된 지역 펫시터를 연결하는 플랫폼'],
 'mustPreserve': ['보호자가 가까운 지역에서 믿을 수 있는 돌봄 제공자를 빠르게 찾고 비교하기 어렵다.',
  '출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자',
  '반려동물 보호자와 검증된 지역 펫시터를 연결하는 플랫폼'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis))

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,반려동물 보호자와 검증된 지역 펫시터를 연결하고 예약·결제·이용 후기를 제공하는 서비스
1,SOURCE_LOCK,problem,보호자가 가까운 지역에서 믿을 수 있는 돌봄 제공자를 빠르게 찾고 비교하기 어렵다.
2,SOURCE_LOCK,targetUsers,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자
3,SEMANTIC_ANCHOR,ideaOverview,반려동물 보호자와 검증된 지역 펫시터를 연결하고 예약·결제·이용 후기를 제공하는 서비스
4,SEMANTIC_ANCHOR,problem,보호자가 가까운 지역에서 믿을 수 있는 돌봄 제공자를 빠르게 찾고 비교하기 어렵다.
5,SEMANTIC_ANCHOR,targetUsers,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
plans = engine._last_plan_pool
print({'totalPlans': len(plans), 'planningRounds': plan_validation.planningRounds,
       'replenishmentRequested': plan_validation.replenishmentRequested})

{'totalPlans': 6, 'planningRounds': 1, 'replenishmentRequested': 0}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status))
display({'planningRounds': plan_validation.planningRounds,
         'replenishmentRequested': plan_validation.replenishmentRequested,
         'adaptiveReplenishmentUsed': plan_validation.planningRounds > 1})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,1,RESERVE_SHORTFALL


{'planningRounds': 1,
 'replenishmentRequested': 0,
 'adaptiveReplenishmentUsed': False}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans))

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P2,단기 펫시터 예약 플랫폼,SELECTED,0.9094,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신속하고 간편한 펫시터 예약으로 보호자의 걱정 해소,단기 돌봄 필요에 최적화된 펫시터 예약 서비스 제공,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 구축,"{'businessRole': 'OTHER', 'operatingModel': 'P...",출장 중에도 반려동물 돌봄을 원활하게 제공하기 위해.
1,P5,신뢰성 높은 펫시터 추천 서비스,SELECTED,0.8044,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신뢰할 수 있는 펫시터를 추천하여 보호자의 걱정 해소,검증된 펫시터를 추천하여 보호자의 신뢰를 구축한다.,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",신뢰할 수 있는 펫시터 추천으로 보호자의 불안 해소.
2,P3,지역 기반 펫시터 매칭 서비스,SELECTED,0.7244,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,지역 기반의 신뢰할 수 있는 펫시터를 쉽게 찾을 수 있는 서비스 제공,지역 내 검증된 펫시터와 보호자를 연결하여 신뢰성을 높인다.,사용자가 쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",지역 내 신뢰할 수 있는 펫시터를 통해 보호자의 불안 해소.
3,P4,펫시터 예약 및 후기 플랫폼,SELECTED,0.6444,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,펫시터 예약과 후기를 한 곳에서 관리할 수 있는 편리함 제공,펫시터 예약과 이용 후기를 통합하여 사용자 편의성 증대,사용자가 손쉽게 펫시터를 예약하고 후기를 남길 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",펫시터 예약과 후기를 통합하여 사용자 편의성을 높이기 위해.
4,P6,펫시터와 보호자 간의 소통 플랫폼,SELECTED,0.5481,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,펫시터와의 직접 소통으로 신뢰성 증대,펫시터와 보호자가 직접 소통할 수 있는 플랫폼 제공,사용자가 손쉽게 펫시터와 소통하고 예약할 수 있는 서비스 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",펫시터와 보호자 간의 신뢰를 구축하기 위해.
5,P1,신뢰할 수 있는 펫시터 연결 서비스,RESERVE,0.7312,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 파트너 네트워크,출장이나 야근으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신뢰할 수 있는 펫시터를 쉽게 찾고 예약할 수 있는 편리함 제공,지역 기반의 검증된 펫시터와 보호자를 연결하여 신뢰성을 높인다.,사용자가 간편하게 펫시터를 검색하고 예약할 수 있는 직관적인 인터페이스 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...","반려동물 보호자와 펫시터 간의 신뢰를 구축하고, 편리한 서비스를 제공하기 위해."


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]})

{'accepted': ['P2', 'P5', 'P3', 'P4', 'P6'], 'rejected': []}

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity))

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, valuePropositionThesis, o...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
1,P1,P3,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, valuePropositionThesis, s...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
2,P2,P3,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
3,P1,P4,DISTINCT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, valuePropositionThesis, o...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,P2,P4,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
5,P3,P4,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
6,P1,P5,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","targetSegmentThesis, valuePropositionThesis, o...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
7,P2,P5,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,P3,P5,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
9,P4,P5,VARIANT,OTHER:PARTNER_NETWORK,OTHER:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans
reserve_plans = plan_validation.reservePlans
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P2,단기 펫시터 예약 플랫폼,SELECTED,0.9094,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신속하고 간편한 펫시터 예약으로 보호자의 걱정 해소,단기 돌봄 필요에 최적화된 펫시터 예약 서비스 제공,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 구축,"{'businessRole': 'OTHER', 'operatingModel': 'P...",출장 중에도 반려동물 돌봄을 원활하게 제공하기 위해.
1,P5,신뢰성 높은 펫시터 추천 서비스,SELECTED,0.8044,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신뢰할 수 있는 펫시터를 추천하여 보호자의 걱정 해소,검증된 펫시터를 추천하여 보호자의 신뢰를 구축한다.,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",신뢰할 수 있는 펫시터 추천으로 보호자의 불안 해소.
2,P3,지역 기반 펫시터 매칭 서비스,SELECTED,0.7244,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,지역 기반의 신뢰할 수 있는 펫시터를 쉽게 찾을 수 있는 서비스 제공,지역 내 검증된 펫시터와 보호자를 연결하여 신뢰성을 높인다.,사용자가 쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",지역 내 신뢰할 수 있는 펫시터를 통해 보호자의 불안 해소.
3,P4,펫시터 예약 및 후기 플랫폼,SELECTED,0.6444,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,펫시터 예약과 후기를 한 곳에서 관리할 수 있는 편리함 제공,펫시터 예약과 이용 후기를 통합하여 사용자 편의성 증대,사용자가 손쉽게 펫시터를 예약하고 후기를 남길 수 있는 플랫폼 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",펫시터 예약과 후기를 통합하여 사용자 편의성을 높이기 위해.
4,P6,펫시터와 보호자 간의 소통 플랫폼,SELECTED,0.5481,같은 Family를 허용하면서 의미 있는 target/use/value Varian...,VARIANT,기타 역할 · 파트너 네트워크,출장·야근 등으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,펫시터와의 직접 소통으로 신뢰성 증대,펫시터와 보호자가 직접 소통할 수 있는 플랫폼 제공,사용자가 손쉽게 펫시터와 소통하고 예약할 수 있는 서비스 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...",펫시터와 보호자 간의 신뢰를 구축하기 위해.
5,P1,신뢰할 수 있는 펫시터 연결 서비스,RESERVE,0.7312,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 파트너 네트워크,출장이나 야근으로 단기 돌봄이 필요한 반려동물 보호자,반려동물 돌봄 서비스 제공,신뢰할 수 있는 펫시터를 쉽게 찾고 예약할 수 있는 편리함 제공,지역 기반의 검증된 펫시터와 보호자를 연결하여 신뢰성을 높인다.,사용자가 간편하게 펫시터를 검색하고 예약할 수 있는 직관적인 인터페이스 제공,"{'businessRole': 'OTHER', 'operatingModel': 'P...","반려동물 보호자와 펫시터 간의 신뢰를 구축하고, 편리한 서비스를 제공하기 위해."


{'selected': ['P2', 'P5', 'P3', 'P4', 'P6'], 'reserve': ['P1']}

## 19. Candidate 1


In [20]:
candidate_one = await engine.expand_plan(seed, selected_plans[0], 1) if selected_plans else None
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,단기 펫시터 예약 플랫폼,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 구축,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,INTERMEDIARY:PARTNER_NETWORK,partnerModel,OTHER,LOW,UNKNOWN
3,C1,INTERMEDIARY:PARTNER_NETWORK,deliveryModel,ON_SITE,HIGH,RULE
4,C1,INTERMEDIARY:PARTNER_NETWORK,transactionModel,BOOKING,HIGH,RULE
5,C1,INTERMEDIARY:PARTNER_NETWORK,monetizationModel,COMMISSION,HIGH,RULE
6,C1,INTERMEDIARY:PARTNER_NETWORK,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:PARTNER_NETWORK,dataDependency,MATERIAL,NaN,NaN
8,C1,INTERMEDIARY:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
for i, plan in enumerate(selected_plans[1:], 2):
    remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,단기 펫시터 예약 플랫폼,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 구축,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영
1,C2,L2,None,신뢰성 높은 펫시터 추천 서비스,사용자가 손쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영
2,C3,L3,None,지역 기반 펫시터 매칭 서비스,사용자가 쉽게 펫시터를 검색하고 예약할 수 있는 플랫폼 제공,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영
3,C4,L4,None,펫시터 예약 및 후기 플랫폼,사용자가 손쉽게 펫시터를 예약하고 후기를 남길 수 있는 플랫폼 제공,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영
4,C5,L5,None,펫시터와 보호자 간의 소통 플랫폼,사용자가 손쉽게 펫시터와 소통하고 예약할 수 있는 서비스 제공,거래 중개 · 파트너 네트워크,{'thesis': {'targetSegmentThesis': '출장·야근 등으로 ...,서비스 이용 수수료를 통한 수익 모델,지역 펫시터와의 파트너십을 통해 서비스 운영


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:PARTNER_NETWORK,operatingModel,PARTNER_NETWORK,HIGH,RULE
2,C1,INTERMEDIARY:PARTNER_NETWORK,partnerModel,OTHER,LOW,UNKNOWN
3,C1,INTERMEDIARY:PARTNER_NETWORK,deliveryModel,ON_SITE,HIGH,RULE
4,C1,INTERMEDIARY:PARTNER_NETWORK,transactionModel,BOOKING,HIGH,RULE
5,C1,INTERMEDIARY:PARTNER_NETWORK,monetizationModel,COMMISSION,HIGH,RULE
6,C1,INTERMEDIARY:PARTNER_NETWORK,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:PARTNER_NETWORK,dataDependency,MATERIAL,NaN,NaN
8,C1,INTERMEDIARY:PARTNER_NETWORK,physicalDependency,MATERIAL,NaN,NaN
9,C2,INTERMEDIARY:PARTNER_NETWORK,businessRole,INTERMEDIARY,HIGH,RULE


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
candidates = candidate_preparation.candidates
candidate_reports = candidate_preparation.reports
display(show_candidate_recovery(candidate_preparation))
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   5                           5                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         0   
 
    finalCandidatePortfolio  
 0                        5  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 4          C5         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS          ADAPTED                  True   
 1        

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
1,C1,C3,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
2,C1,C4,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
3,C1,C5,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
4,C2,C3,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
5,C2,C4,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
6,C2,C5,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
7,C3,C4,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,C3,C5,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
9,C4,C5,VARIANT,INTERMEDIARY:PARTNER_NETWORK,INTERMEDIARY:PARTNER_NETWORK,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates]
display(show_legal_precheck(prechecks))
legal_preparation = await engine.prepare_legal_candidates(seed, candidates)
candidates_before_legal = candidates
candidates = legal_preparation.candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports],
         'completionAttempted': legal_preparation.completionAttempted,
         'completionValidated': legal_preparation.completionValidated,
         'completionAccepted': legal_preparation.completionAccepted,
         'completionExhausted': legal_preparation.completionExhausted,
         'excludedCandidates': legal_preparation.excludedCandidates})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,False,False,True,True,True,"[물리 활동, 개인정보, 자격 의존]"
1,C2,Structural risk precheck — not final legal review,False,False,True,True,False,"[물리 활동, 개인정보]"
2,C3,Structural risk precheck — not final legal review,False,False,True,True,False,"[물리 활동, 개인정보]"
3,C4,Structural risk precheck — not final legal review,False,False,True,True,False,"[물리 활동, 개인정보]"
4,C5,Structural risk precheck — not final legal review,False,True,True,True,False,"[물리 활동, 개인정보]"


{'factCompleteness': [{'candidateId': 'C1',
   'status': 'COMPLETABLE',
   'missingDesignFacts': ['providerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.',
    'sellerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.',
    'intermediaryRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'contradictions': [],
   'completionRequirements': ['providerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.',
    'sellerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.',
    'intermediaryRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'affectedFields': ['providerRole', 'sellerRole', 'intermediaryRole'],
   'roleSemantics': [{'field': 'platformRole',
     'status': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'providerRole',
     'status': 'AMBIGUOUS',
     'safeReason': '해당 역할의 주체와 책임이 불명확합니다.'},
    {'field': 'sellerRole',
     'status': 'AMBIGUOUS',
     'safeReason': '해당 역할의 주체와 책임이 불명확합니다.'},
    {'field': 'intermediaryRole',
     'status': 'AMBIGUOUS',
     'safeReason': '해당 역할의 주체와 책임이 불명확합니다.'}],
   'safeSummary': '

[]

## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter()
legal_c1_input = legal_adapter.task_input(candidates[0].candidate, seed) if candidates else None
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': None,
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

{'initialRoute': 'SKIPPED',
 'redesignRequirements': [],
 'recoveryReviews': [],
 'requiredInputs': [],
 'terminalCandidates': 0,
 'redesigned': 0,
 'replanned': 0}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = LIVE_TEST_LEVEL == 'FULL_E2E'
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated,
       'Candidate Accepted': len(candidates),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.


{'recoveryTrace': [],
 'requiredInputs': [],
 'metrics': {'factAttempted': 5,
  'factValidated': 5,
  'factAccepted': 0,
  'factExhausted': 5,
  'redesignAttempted': 0,
  'redesignValidated': 0,
  'redesignAccepted': 0,
  'redesignExhausted': 0,
  'replanAttempted': 0,
  'replanValidated': 0,
  'replanAccepted': 0,
  'replanExhausted': 0}}

{'Plan Selected': 5, 'Candidate Generated': 5, 'Candidate Accepted': 0, 'Legal Accepted': 0, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 0}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 1}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

{'status': 'LEGAL_PENDING'}

## 33. Unresolved Candidate Summary


In [34]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

{'unresolved': []}

## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': None}


## 35. 7 Hypotheses


In [36]:
hypotheses = engine.build_or_load_current_hypothesis_contract(selected_concept) if selected_concept else []
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

""


{'All Hypotheses Semantically Ready': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

""


{'All Hypotheses Semantically Ready': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['All Hypotheses Semantically Ready']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'status': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'NOT_READY', 'reason': None, 'unresolvedHypotheses': []}

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T03:44:12.474876+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,None
1,2,2026-08-10T03:44:19.783937+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,None
2,3,2026-08-10T03:44:19.783957+00:00,SAFETY_CHECKING,READINESS_INCONSISTENT,NaN,NaN,WARNING,LIVE,NaN,READY_FOR_REVIEW이지만 score=0입니다. V2 gating은 막지 ...,READINESS_INCONSISTENT,None
3,4,2026-08-10T03:44:19.857493+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,None
4,5,2026-08-10T03:44:19.857724+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,None
5,6,2026-08-10T03:44:20.065358+00:00,PLANNING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,최대 5개 동적 plan을 요청합니다.,NaN,None
6,7,2026-08-10T03:44:48.422436+00:00,PLANNING,DRAFTS_GENERATED,NaN,NaN,PASS,LIVE,2.0,Plan draft pool=6,NaN,None
7,8,2026-08-10T03:44:48.424080+00:00,PLANNING,NORMALIZED,NaN,NaN,PASS,LIVE,NaN,System metadata를 부여한 Plan=6,NaN,None
8,9,2026-08-10T03:44:54.709367+00:00,PLANNING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,NaN,PASS,LIVE,3.0,low-confidence Plan architecture 6개를 batch 분류했...,NaN,None
9,10,2026-08-10T03:44:54.709413+00:00,PLAN_VALIDATING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,Opportunity·LOCK·DUPLICATE/VARIANT/DISTINCT 관계...,NaN,None


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,13,13,"{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...",0,139593,{'LIVE': 13},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                    operation  \
 0                  PLAN_POOL   
 1                  PLAN_POOL   
 2          SEMANTIC_RELATION   
 3    NORMALIZE_ARCHITECTURES   
 4      IDEA_BRIEF_DERIVATION   
 ..                       ...   
 180    IDEA_BRIEF_DERIVATION   
 181                   EXPAND   
 182                   EXPAND   
 183                 REDESIGN   
 184                   EXPAND   
 
                                                   hash operationVersion  \
 0    038e1d9753b8c07ad835823495b54906a14fb6542c3c70...               v3   
 1    03b73ea7d485636893670a848e28d49e239cc8bc0c5239...             v2.1   
 2    05680e1eabb667cd53d8bbde75192650b626fd5dfaa3fd...               v3   
 3    07ba75e844ed104c5fc7d25bf69f296244d0925e2f8916...               v1   
 4    07c6209c7795158bce16bf1d386c8f6cd1322277337ef9...               v2   
 ..                                                 ...              ...   
 180  f4efc411e5d763112705c19d

## 44. One-click MOCK


In [45]:
mock_result = await ConceptPortfolioEngine('MOCK').run_full(
    TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result))
assert mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS'

,runId,runStatus,runtimeStage,producedConceptCount,downstreamReadiness,safety,requestedMaximum,planned,planSelected,planDuplicatesRemoved,...,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs,failureStage,failureCode
0,409eef15-0b7e-4a91-aaf5-2eea179714c4,READY_FULL,READY,5,PASS,PASS,5,7,5,0,...,0,0,5,READY_FULL,직접 운영 핵심형,PASS,0,93,None,None


## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})

,runId,runStatus,runtimeStage,producedConceptCount,downstreamReadiness,safety,requestedMaximum,planned,planSelected,planDuplicatesRemoved,...,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs,failureStage,failureCode
0,3eec4913-8450-46dc-8a22-71a7a38841e0,FAILED,FAILED,0,INVALID,PASS,5,6,5,0,...,0,0,0,FAILED,None,INVALID,23,160099,FAILED,None


,Scenario,Plan returned,Plan selected,Candidate valid,Legal ready,Legal ACCEPT,NEEDS_INPUT,Final portfolio,Hypothesis valid,Handoff,Provider ops,Duration(ms)
0,LOCAL_PETCARE_MARKETPLACE,6,5,10,0,0,0,0,INVALID,PENDING,23,160099


,candidateId,scope,unknownFacts,reason,possibleUserAction,currentValue,requiredLegalChange,safeSummary
0,C1-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
1,C2-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
2,C3-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
3,C4-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.
4,C5-F1,CANDIDATE,None,None,None,None,None,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.


,failedStage,failureCode,safeSummary,failedEntityId,providerFailure,lastSuccessfulStage,firstFailedStage,lastTraceEvents
0,FAILED,UNCLASSIFIED_SYSTEM_FAILURE,최종 Portfolio=0,None,{},CANDIDATE_VALIDATING,FAILED,20


{'상태': '기록된 Provider 실패 없음'}

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,23,23,"{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...",0,160026,{'LIVE': 23},None,None


,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T03:48:34.468947+00:00,CANDIDATE_VALIDATING,STARTED,NaN,None,RUNNING,LIVE,NaN,Candidate 검사를 분리 수행합니다.,NaN,NaN
1,2,2026-08-10T03:48:39.535783+00:00,CANDIDATE_VALIDATING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,None,PASS,LIVE,17.0,low-confidence Candidate architecture 1개를 batc...,NaN,NaN
2,3,2026-08-10T03:48:39.536335+00:00,CANDIDATE_VALIDATING,FIDELITY_AMBIGUOUS,C2-F1,None,RUNNING,LIVE,NaN,lexical 판정이 불확실해 semantic fidelity를 요청합니다.,NaN,CANDIDATE_FIDELITY_RECOVERABLE
3,4,2026-08-10T03:48:42.084887+00:00,CANDIDATE_VALIDATING,SEMANTIC_FIDELITY_CHECKED,C2-F1,None,PASS,LIVE,18.0,"서비스는 반려동물 보호자와 검증된 지역 펫시터를 연결하며, 신속하고 간편한 예약 시...",PASS,NaN
4,5,2026-08-10T03:48:42.085389+00:00,CANDIDATE_VALIDATING,VALIDATED,C2-F1,None,PASS,LIVE,NaN,검사 통과,NaN,NaN
5,6,2026-08-10T03:48:42.086271+00:00,LEGAL_RECOVERING,LEGAL_FACT_COMPLETENESS_CHECKED,C3,None,COMPLETABLE,LIVE,NaN,동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.,COMPLETABLE,NaN
6,7,2026-08-10T03:48:52.243280+00:00,CANDIDATE_VALIDATING,STARTED,NaN,None,RUNNING,LIVE,NaN,Candidate 검사를 분리 수행합니다.,NaN,NaN
7,8,2026-08-10T03:48:52.243714+00:00,CANDIDATE_VALIDATING,FIDELITY_AMBIGUOUS,C3-F1,None,RUNNING,LIVE,NaN,lexical 판정이 불확실해 semantic fidelity를 요청합니다.,NaN,CANDIDATE_FIDELITY_RECOVERABLE
8,9,2026-08-10T03:48:54.953616+00:00,CANDIDATE_VALIDATING,SEMANTIC_FIDELITY_CHECKED,C3-F1,None,PASS,LIVE,20.0,"서비스는 반려동물 보호자와 검증된 지역 펫시터를 연결하며, 리뷰 기반 추천 시스템을...",PASS,NaN
9,10,2026-08-10T03:48:54.954116+00:00,CANDIDATE_VALIDATING,VALIDATED,C3-F1,None,PASS,LIVE,NaN,검사 통과,NaN,NaN


{'unresolvedCandidates': [{'candidateId': 'C1-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C2-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C3-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C4-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'},
  {'candidateId': 'C5-F1',
   'scope': 'CANDIDATE',
   'reasonCode': 'LEGAL_FACT_COMPLETION_EXHAUSTED',
   'safeSummary': '동일 Concept 안에서 누락된 사업 사실을 한 번 보완할 수 있습니다.'}],
 'lastSuccessfulStage': 'CANDIDATE_VALIDATING',
 'firstFailedStage': 'FAILED'}